In [1]:
import urllib.request
import os
from datetime import datetime
import pandas as pd

In [2]:
# Завантаження VHI даних для всіх областей України
def download_vhi_data(save_dir="vhi_data"):
    os.makedirs(save_dir, exist_ok=True)
    
    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={}&year1=1981&year2=2024&type=Mean"
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for province_id in range(1, 28):  # 27 областей України
        filename = f"vhi_province_{province_id}_{timestamp}.csv"
        filepath = os.path.join(save_dir, filename)
        
        # Перевірка чи вже є файл для цієї області
        existing = [f for f in os.listdir(save_dir) if f.startswith(f"vhi_province_{province_id}_")]
        if existing:
            print(f"Область {province_id}: файл вже існує, пропускаємо")
            continue
        
        url = base_url.format(province_id)
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Область {province_id}: завантажено ✓")
        except Exception as e:
            print(f"Область {province_id}: помилка — {e}")

download_vhi_data()

Область 1: файл вже існує, пропускаємо
Область 2: файл вже існує, пропускаємо
Область 3: файл вже існує, пропускаємо
Область 4: файл вже існує, пропускаємо
Область 5: файл вже існує, пропускаємо
Область 6: файл вже існує, пропускаємо
Область 7: файл вже існує, пропускаємо
Область 8: файл вже існує, пропускаємо
Область 9: файл вже існує, пропускаємо
Область 10: файл вже існує, пропускаємо
Область 11: файл вже існує, пропускаємо
Область 12: файл вже існує, пропускаємо
Область 13: файл вже існує, пропускаємо
Область 14: файл вже існує, пропускаємо
Область 15: файл вже існує, пропускаємо
Область 16: файл вже існує, пропускаємо
Область 17: файл вже існує, пропускаємо
Область 18: файл вже існує, пропускаємо
Область 19: файл вже існує, пропускаємо
Область 20: файл вже існує, пропускаємо
Область 21: файл вже існує, пропускаємо
Область 22: файл вже існує, пропускаємо
Область 23: файл вже існує, пропускаємо
Область 24: файл вже існує, пропускаємо
Область 25: файл вже існує, пропускаємо
Область 2

In [3]:
def load_vhi_data(data_dir="vhi_data"):
    frames = []
    
    for filename in os.listdir(data_dir):
        if not filename.endswith(".csv"):
            continue
        
        province_id = int(filename.split("_")[2])
        filepath = os.path.join(data_dir, filename)
        
        # Читаємо і чистимо сирий файл
        lines = []
        with open(filepath, 'r') as f:
            for line in f:
                # Прибираємо HTML теги та сміття
                line = line.replace('<br>', '').replace('<tt><pre>', '').replace('</tt></pre>', '').strip()
                # Пропускаємо заголовки та порожні рядки
                if not line or line.startswith('Mean') or line.startswith('year') or line.startswith('<'):
                    continue
                # Прибираємо кінцеву кому
                line = line.rstrip(',')
                lines.append(line)
        
        # Парсимо як csv з наших очищених рядків
        from io import StringIO
        clean_csv = "year,week,SMN,SMT,VCI,TCI,VHI\n" + "\n".join(lines)
        df = pd.read_csv(StringIO(clean_csv), skipinitialspace=True)
        
        # Видаляємо рядки де VHI == -1
        df = df[df['VHI'] != -1]
        
        # Додаємо province_id
        df['province_id'] = province_id
        
        frames.append(df)
    
    result = pd.concat(frames, ignore_index=True)
    return result

df = load_vhi_data()
df.head(10)

,year,week,SMN,SMT,VCI,TCI,VHI,province_id
0,1982,1,0.038,252.22,47.05,69.15,58.10,20260308
1,1982,2,0.036,253.39,45.39,63.52,54.45,20260308
2,1982,3,0.033,254.22,41.22,60.22,50.72,20260308
3,1982,4,0.031,255.60,37.35,58.01,47.68,20260308
4,1982,5,0.030,257.16,32.54,56.92,44.73,20260308
5,1982,6,0.028,258.12,25.63,57.18,41.41,20260308
6,1982,7,0.029,259.90,21.30,54.76,38.03,20260308
7,1982,8,0.033,262.82,21.84,49.76,35.80,20260308
8,1982,9,0.036,265.27,20.87,48.45,34.66,20260308
9,1982,10,0.041,267.96,20.64,48.64,34.64,20260308


In [4]:
# Маппінг: NOAA province_id -> назва області (англійська абетка)
noaa_to_name = {
    1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea",
    5: "Dnipropetrovsk", 6: "Donetsk", 7: "Ivano-Frankivsk", 8: "Kharkiv",
    9: "Kherson", 10: "Khmelnytskyi", 11: "Kyiv", 12: "Kyiv City",
    13: "Kirovohrad", 14: "Luhansk", 15: "Lviv", 16: "Mykolaiv",
    17: "Odessa", 18: "Poltava", 19: "Rivne", 20: "Sumy",
    21: "Ternopil", 22: "Vinnytsia", 23: "Volyn", 24: "Zakarpattia",
    25: "Zaporizhzhia", 26: "Zhytomyr", 27: "Sevastopol"
}

# Маппінг: назва -> новий індекс за українською абеткою
ua_index = {
    "Vinnytsia": 1, "Volyn": 2, "Dnipropetrovsk": 3, "Donetsk": 4,
    "Zhytomyr": 5, "Zakarpattia": 6, "Zaporizhzhia": 7, "Ivano-Frankivsk": 8,
    "Kyiv": 9, "Kyiv City": 10, "Kirovohrad": 11, "Crimea": 12,
    "Luhansk": 13, "Lviv": 14, "Mykolaiv": 15, "Odessa": 16,
    "Poltava": 17, "Rivne": 18, "Sumy": 19, "Ternopil": 20,
    "Kharkiv": 21, "Kherson": 22, "Khmelnytskyi": 23, "Cherkasy": 24,
    "Chernivtsi": 25, "Chernihiv": 26, "Sevastopol": 27
}

# Додаємо стовпці
df['region_name'] = df['province_id'].map(noaa_to_name)
df['region_id_ua'] = df['region_name'].map(ua_index)

df.head(10)

,year,week,SMN,SMT,VCI,TCI,VHI,province_id,region_name,region_id_ua
0,1982,1,0.038,252.22,47.05,69.15,58.10,20260308,NaN,NaN
1,1982,2,0.036,253.39,45.39,63.52,54.45,20260308,NaN,NaN
2,1982,3,0.033,254.22,41.22,60.22,50.72,20260308,NaN,NaN
3,1982,4,0.031,255.60,37.35,58.01,47.68,20260308,NaN,NaN
4,1982,5,0.030,257.16,32.54,56.92,44.73,20260308,NaN,NaN
5,1982,6,0.028,258.12,25.63,57.18,41.41,20260308,NaN,NaN
6,1982,7,0.029,259.90,21.30,54.76,38.03,20260308,NaN,NaN
7,1982,8,0.033,262.82,21.84,49.76,35.80,20260308,NaN,NaN
8,1982,9,0.036,265.27,20.87,48.45,34.66,20260308,NaN,NaN
9,1982,10,0.041,267.96,20.64,48.64,34.64,20260308,NaN,NaN


In [9]:
import urllib.request
import os
from datetime import datetime
import pandas as pd
from io import StringIO

def load_vhi_data(data_dir="vhi_data"):
    frames = []
    
    for filename in os.listdir(data_dir):
        if not filename.endswith(".csv"):
            continue
        
        parts = filename.replace('.csv', '').split('_')
        if 'province' in parts:
            province_id = int(parts[parts.index('province') + 1])
        else:
            province_id = int(parts[1])
        
        filepath = os.path.join(data_dir, filename)
        
        lines = []
        with open(filepath, 'r') as f:
            for line in f:
                line = line.replace('<br>', '').replace('<tt><pre>', '').replace('</tt></pre>', '').strip()
                if not line or line.startswith('Mean') or line.startswith('year') or line.startswith('<'):
                    continue
                line = line.rstrip(',')
                lines.append(line)
        
        clean_csv = "year,week,SMN,SMT,VCI,TCI,VHI\n" + "\n".join(lines)
        df = pd.read_csv(StringIO(clean_csv), skipinitialspace=True)
        df = df[df['VHI'] != -1]
        df['province_id'] = province_id
        frames.append(df)
    
    return pd.concat(frames, ignore_index=True)

noaa_to_name = {
    1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea",
    5: "Dnipropetrovsk", 6: "Donetsk", 7: "Ivano-Frankivsk", 8: "Kharkiv",
    9: "Kherson", 10: "Khmelnytskyi", 11: "Kyiv", 12: "Kyiv City",
    13: "Kirovohrad", 14: "Luhansk", 15: "Lviv", 16: "Mykolaiv",
    17: "Odessa", 18: "Poltava", 19: "Rivne", 20: "Sumy",
    21: "Ternopil", 22: "Vinnytsia", 23: "Volyn", 24: "Zakarpattia",
    25: "Zaporizhzhia", 26: "Zhytomyr", 27: "Sevastopol"
}

ua_index = {
    "Vinnytsia": 1, "Volyn": 2, "Dnipropetrovsk": 3, "Donetsk": 4,
    "Zhytomyr": 5, "Zakarpattia": 6, "Zaporizhzhia": 7, "Ivano-Frankivsk": 8,
    "Kyiv": 9, "Kyiv City": 10, "Kirovohrad": 11, "Crimea": 12,
    "Luhansk": 13, "Lviv": 14, "Mykolaiv": 15, "Odessa": 16,
    "Poltava": 17, "Rivne": 18, "Sumy": 19, "Ternopil": 20,
    "Kharkiv": 21, "Kherson": 22, "Khmelnytskyi": 23, "Cherkasy": 24,
    "Chernivtsi": 25, "Chernihiv": 26, "Sevastopol": 27
}

df = load_vhi_data()
df['region_name'] = df['province_id'].map(noaa_to_name)
df['region_id_ua'] = df['region_name'].map(ua_index)

df.head(10)

,year,week,SMN,SMT,VCI,TCI,VHI,province_id,region_name,region_id_ua
0,1982,1,0.038,252.22,47.05,69.15,58.10,8,Kharkiv,21
1,1982,2,0.036,253.39,45.39,63.52,54.45,8,Kharkiv,21
2,1982,3,0.033,254.22,41.22,60.22,50.72,8,Kharkiv,21
3,1982,4,0.031,255.60,37.35,58.01,47.68,8,Kharkiv,21
4,1982,5,0.030,257.16,32.54,56.92,44.73,8,Kharkiv,21
5,1982,6,0.028,258.12,25.63,57.18,41.41,8,Kharkiv,21
6,1982,7,0.029,259.90,21.30,54.76,38.03,8,Kharkiv,21
7,1982,8,0.033,262.82,21.84,49.76,35.80,8,Kharkiv,21
8,1982,9,0.036,265.27,20.87,48.45,34.66,8,Kharkiv,21
9,1982,10,0.041,267.96,20.64,48.64,34.64,8,Kharkiv,21


In [10]:
# 1. Ряд VHI для області за вказаний рік
def vhi_by_region_year(df, region_id_ua, year):
    result = df[(df['region_id_ua'] == region_id_ua) & (df['year'] == year)][['year', 'week', 'VHI', 'region_name']]
    print(f"VHI для області {region_id_ua} ({result['region_name'].iloc[0]}) за {year} рік:")
    return result

vhi_by_region_year(df, 21, 2000)

VHI для області 21 (Kharkiv) за 2000 рік:


,year,week,VHI,region_name
906,2000,1,30.98,Kharkiv
907,2000,2,31.55,Kharkiv
908,2000,3,33.38,Kharkiv
909,2000,4,35.60,Kharkiv
910,2000,5,37.97,Kharkiv
...,...,...,...,...
18441,2000,48,9.36,Kharkiv
18442,2000,49,9.45,Kharkiv
18443,2000,50,9.73,Kharkiv
18444,2000,51,11.45,Kharkiv


In [11]:
# 2. Ряд VHI за діапазон років для вказаних областей
def vhi_by_regions_years(df, region_ids, year_start, year_end):
    result = df[(df['region_id_ua'].isin(region_ids)) & 
                (df['year'] >= year_start) & 
                (df['year'] <= year_end)][['year', 'week', 'VHI', 'region_name', 'region_id_ua']]
    print(f"VHI для областей {region_ids} за {year_start}-{year_end}:")
    return result

vhi_by_regions_years(df, [21, 1], 2000, 2005)

VHI для областей [21, 1] за 2000-2005:


,year,week,VHI,region_name,region_id_ua
906,2000,1,30.98,Kharkiv,21
907,2000,2,31.55,Kharkiv,21
908,2000,3,33.38,Kharkiv,21
909,2000,4,35.60,Kharkiv,21
910,2000,5,37.97,Kharkiv,21
...,...,...,...,...,...
49285,2005,48,46.67,Vinnytsia,1
49286,2005,49,49.46,Vinnytsia,1
49287,2005,50,49.87,Vinnytsia,1
49288,2005,51,48.88,Vinnytsia,1


In [12]:
# 3. Екстремуми, середнє, медіана
def vhi_stats(df, region_ids, year_start, year_end):
    subset = df[(df['region_id_ua'].isin(region_ids)) & 
                (df['year'] >= year_start) & 
                (df['year'] <= year_end)]
    
    stats = subset.groupby('region_name')['VHI'].agg(
        мінімум='min',
        максимум='max',
        середнє='mean',
        медіана='median'
    ).round(2)
    
    print(f"Статистика VHI для областей {region_ids} за {year_start}-{year_end}:")
    return stats

vhi_stats(df, [21, 1, 5], 2000, 2010)

Статистика VHI для областей [21, 1, 5] за 2000-2010:


,мінімум,максимум,середнє,медіана
region_name,,,,
Kharkiv,9.36,91.42,50.50,49.13
Vinnytsia,24.33,75.04,50.03,50.14
Zhytomyr,10.88,87.30,46.39,45.16
